# Lab 5: Simple Tool‑Choosing Agent (Calculator vs RAG)

<a href="https://colab.research.google.com/github/Tulane-CMPS-1010-AI-Systems/course-materials/blob/main/labs/05-agents_lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

---

## Overview
An **agent** is a control loop: **decide → act (tool call) → observe → return**.

In this lab you will:
1. Implement a **tool-selection policy** (simple Python rules).
2. Wire the policy into a one-step **`agent_step`** function that calls tools and logs results.
3. Run a small **experiment** comparing two policies and measuring how tool choice affects accuracy.

In [ ]:
# @title 🔧 Setup (Run this first)
!git clone --depth 1 -q https://github.com/Tulane-CMPS-1010-AI-Systems/course-materials.git
import sys; sys.path.append('./course-materials')

from course_utils import (
    lab6_setup,
    lab6_get_corpus,
    lab6_build_retriever,
    lab6_rag_retrieve,
    lab6_calculator,
    lab6_generate_answer,
    lab6_build_demo,
    lab6_default_eval_set,
)

lab6_setup()
print("✅ Environment ready!")

**--- TODO**

## Pre-Lab Questions
Answer in **1–2 sentences each**.

1. When should an agent call a **calculator tool** instead of answering directly?
2. When should an agent use **RAG** (retrieval) instead of relying on the model’s memory?
3. What could go wrong if an agent loops without a stop condition?

**Your answers:**
1)  
2)  
3)

**--- TODO**

## Hypothesis

We’ll compare two tool-selection policies:
- **Policy A** (baseline): only detects math → `calculator`, else → `none`.
- **Policy B** (improved): detects math → `calculator`, detects doc keywords → `rag`, else → `none`.

**Question:** If we change the tool-selection policy, what happens to tool-choice accuracy and answer quality?

Write your hypothesis (2–4 sentences):

> **My hypothesis:**  
> …

---
# Part 1 — Meet the tools
Our agent has two tools (plain Python functions):

| Tool | Call | Returns |
|------|------|---------|
| Calculator | `lab6_calculator(expr)` | `{"result": 85}` or `{"error": "..."}` |
| RAG retrieve | `lab6_rag_retrieve(query, retriever)` | `{"passages": [...], "scores": [...]}` |

Try each one below so you see what the outputs look like.

In [ ]:
lab6_calculator("17 * (2 + 3)")

In [ ]:
corpus = lab6_get_corpus()
retriever = lab6_build_retriever(corpus=corpus, chunk_size=60, overlap=15, top_k=4)

lab6_rag_retrieve(query="According to the policy, can interns join on-call?", retriever=retriever)

---
# Part 2 — Implement two tool-selection policies

### Policy A (baseline)
- If the question contains digits **and** math operators → `'calculator'`
- Else → `'none'`

### Policy B (improved)
- If it looks like math → `'calculator'`
- Else if it mentions docs/policy/runbook → `'rag'`
- Else → `'none'`

In [ ]:
# @title ✅ TODO: Implement tool-selection helpers
import re

def looks_like_math(question: str) -> bool:
    """
    Return True if the question contains a math expression.

    Hints:
    - math questions contain digits (0-9)
    - and operators like + - * / % ( )
    """
    # TODO: replace the next line with your code
    raise NotImplementedError("Implement looks_like_math(question)")

def looks_like_docs_question(question: str) -> bool:
    """
    Return True if the question wants evidence from docs/policy/runbook.

    Hints:
    - Look for keywords like:
      'according to', 'policy', 'runbook', 'docs', 'documentation', 'source'
    """
    # TODO: replace the next line with your code
    raise NotImplementedError("Implement looks_like_docs_question(question)")

def choose_tool_policy_A(question: str) -> str:
    """Policy A: math -> 'calculator', else -> 'none'"""
    # TODO: implement
    raise NotImplementedError("Implement choose_tool_policy_A(question)")

def choose_tool_policy_B(question: str) -> str:
    """Policy B: math -> 'calculator', docs -> 'rag', else -> 'none'"""
    # TODO: implement
    raise NotImplementedError("Implement choose_tool_policy_B(question)")

In [ ]:
# @title (Optional) Reference solution for tool-choice helpers

def looks_like_math(question: str) -> bool:
    has_digit = bool(re.search(r"\d", question))
    has_op = bool(re.search(r"[+\-*/%()]", question))
    return has_digit and has_op

def looks_like_docs_question(question: str) -> bool:
    q = question.lower()
    triggers = ["according to", "policy", "runbook", "docs", "documentation", "source", "what does it say"]
    return any(t in q for t in triggers)

def choose_tool_policy_A(question: str) -> str:
    return "calculator" if looks_like_math(question) else "none"

def choose_tool_policy_B(question: str) -> str:
    if looks_like_math(question):
        return "calculator"
    if looks_like_docs_question(question):
        return "rag"
    return "none"

print("Loaded reference solution ✅")

---
# Part 3 — Build a one-step agent

The `agent_step` function does one loop iteration:
1. **Choose** a tool using your policy
2. **Call** the tool (calculator, RAG, or nothing)
3. **Return** a log dict with the tool choice, output, answer, and trace

Almost everything is provided below — you only fill in **4 lines** (the actual tool calls).

In [ ]:
# @title ✅ TODO: Fill in the 4 tool-call lines in agent_step()
from typing import Callable, Dict, Any, List

def extract_math_expression(question: str) -> str:
    """Extract a rough math expression from the question."""
    expr = re.sub(r"[^0-9+\-*/().% ]", "", question)
    return expr.strip()

def agent_step(question: str,
               policy: Callable[[str], str],
               retriever,
               use_llm_for_final_answer: bool = True) -> Dict[str, Any]:
    """
    Run ONE agent step.  Fill in the 4 TODO lines below.

    Tool functions you can call:
      lab6_calculator(expr)              -> {"result": ...} or {"error": ...}
      lab6_rag_retrieve(query, retriever)-> {"passages": [...], "scores": [...]}
      lab6_generate_answer(question, passages=None) -> str
    """
    tool = policy(question)
    trace = [f"TOOL={tool}"]
    tool_output = None
    answer = ""

    if tool == "calculator":
        expr = extract_math_expression(question)
        trace.append(f"EXPR={expr}")

        # TODO (1 line): call the calculator tool on `expr`
        tool_output = ...   # ← replace ...

        if "result" in tool_output:
            answer = f"The answer is {tool_output['result']}  (computed: {expr})"
        else:
            answer = f"Calculator error: {tool_output.get('error', 'unknown')}"

    elif tool == "rag":
        # TODO (1 line): call the RAG retrieval tool
        rag_out = ...   # ← replace ...

        passages = rag_out.get("passages", [])
        tool_output = rag_out
        trace.append(f"PASSAGES_RETRIEVED={len(passages)}")

        if use_llm_for_final_answer and passages:
            # TODO (1 line): generate an LLM answer from question + passages
            answer = ...   # ← replace ...
        elif passages:
            answer = f"Top passage: {passages[0]}"
        else:
            answer = "No relevant passages found."

    else:  # tool == "none"
        if use_llm_for_final_answer:
            # TODO (1 line): generate an LLM answer with no passages
            answer = ...   # ← replace ...
        else:
            answer = "(No tool used; LLM generation disabled.)"

    return {
        "question": question,
        "tool": tool,
        "tool_output": tool_output,
        "answer": answer,
        "trace": trace,
    }


def run_agent(questions: List[str], policy, retriever, use_llm_for_final_answer=True):
    """Run agent_step on every question and collect the logs."""
    return [agent_step(q, policy=policy, retriever=retriever,
                       use_llm_for_final_answer=use_llm_for_final_answer)
            for q in questions]

---
# Part 4 — Experiment: Policy A vs Policy B

We’ll run both policies on the same 8 labeled questions and measure:
- **Tool-choice accuracy** — did we pick the gold tool?
- **Quality proxy** — did the tool produce a valid output? (calculator returned a result, RAG returned passages, none returned a non-empty answer)

In [ ]:
# @title Build retriever + load eval set
import pandas as pd

corpus = lab6_get_corpus()
retriever = lab6_build_retriever(corpus=corpus, chunk_size=60, overlap=15, top_k=4)

eval_set = lab6_default_eval_set()
questions = [ex["q"] for ex in eval_set]
gold_tools = [ex["gold_tool"] for ex in eval_set]

print(f"Loaded {len(eval_set)} eval questions.")
pd.DataFrame(eval_set)

In [ ]:
# @title Metric helpers (provided)

def tool_choice_accuracy(logs, gold_labels):
    """Fraction of examples where the agent chose the gold tool."""
    correct = sum(1 for out, gold in zip(logs, gold_labels) if out["tool"] == gold)
    return correct / len(gold_labels)

def quality_proxy_score(logs):
    """
    Fraction of examples that produced a 'reasonable' output:
      calculator -> tool_output has 'result' key
      rag        -> tool_output has non-empty 'passages'
      none       -> answer is a non-empty string
    """
    good = 0
    for entry in logs:
        tool = entry.get("tool")
        tool_output = entry.get("tool_output")
        answer = entry.get("answer", "")
        if tool == "calculator":
            if isinstance(tool_output, dict) and "result" in tool_output:
                good += 1
        elif tool == "rag":
            if isinstance(tool_output, dict) and len(tool_output.get("passages", [])) > 0:
                good += 1
        else:
            if answer and len(answer.strip()) > 0:
                good += 1
    return good / len(logs)

def summarize_logs(logs, gold_labels):
    """Return a DataFrame for easy inspection."""
    rows = []
    for out, gold in zip(logs, gold_labels):
        rows.append({
            "question": out.get("question"),
            "pred_tool": out.get("tool"),
            "gold_tool": gold,
            "correct": out.get("tool") == gold,
            "answer_preview": (out.get("answer","")[:80] + "…") if out.get("answer") else ""
        })
    return pd.DataFrame(rows)

In [ ]:
# @title Run the experiment
logs_A = run_agent(questions, policy=choose_tool_policy_A, retriever=retriever)
logs_B = run_agent(questions, policy=choose_tool_policy_B, retriever=retriever)

print("Policy A  |  accuracy:", round(tool_choice_accuracy(logs_A, gold_tools), 3),
      " quality:", round(quality_proxy_score(logs_A), 3))
print("Policy B  |  accuracy:", round(tool_choice_accuracy(logs_B, gold_tools), 3),
      " quality:", round(quality_proxy_score(logs_B), 3))

print("\n--- Policy A ---")
display(summarize_logs(logs_A, gold_tools))
print("\n--- Policy B ---")
display(summarize_logs(logs_B, gold_tools))

---
## (Optional) Gradio demo

In [ ]:
demo = lab6_build_demo(
    policy_fn=choose_tool_policy_B,
    agent_step_fn=agent_step,
    retriever=retriever,
)
demo.launch(debug=False, share=False)

---

**--- TODO**

## Results & Conclusion

Answer each prompt in 2–4 sentences.

### Results
- What was the tool-choice accuracy for Policy A vs Policy B?
- Give one example question where Policy A chose the **wrong** tool. Why did it fail?
- Did the quality proxy differ between the two policies? Why or why not?

Write here:

### Conclusion
- Was your hypothesis supported?
- What would you try next to make the agent more reliable?

Write here:

### Reflection
1. Describe one question where your agent behaved in a surprising way.
2. What is one real-world situation where you would *trust* a tool-choosing agent more than a single model call?

Your answers:
1)  
2)

---

## 🧠 AI Usage Log

| Tool Used | Purpose | Prompt / Context | Verification & Edits |
|------------|----------|------------------|----------------------|
| | | | |

**Summary (2–3 sentences):**  

---

In [ ]:
# @title ✅ Run checks for Lab 6
print("Running checks...")

try:
    assert isinstance(looks_like_math("2+2"), bool)
    assert looks_like_math("2+2") is True
    print("✅ looks_like_math")
except Exception as e:
    print("❌ looks_like_math:", e)

try:
    assert looks_like_docs_question("According to the policy...") is True
    print("✅ looks_like_docs_question")
except Exception as e:
    print("❌ looks_like_docs_question:", e)

try:
    for fn in [choose_tool_policy_A, choose_tool_policy_B]:
        assert fn("What is 2+2?") in ["calculator", "rag", "none"]
    print("✅ choose_tool_policy_*")
except Exception as e:
    print("❌ choose_tool_policy_*:", e)

try:
    _c = lab6_get_corpus()
    _r = lab6_build_retriever(corpus=_c, chunk_size=60, overlap=15, top_k=3)
    _out = agent_step("What is 2+2?", policy=choose_tool_policy_B, retriever=_r, use_llm_for_final_answer=False)
    for k in ["question", "tool", "tool_output", "answer", "trace"]:
        assert k in _out
    assert _out["tool"] in ["calculator", "rag", "none"]
    assert isinstance(_out["trace"], list)
    print("✅ agent_step")
except Exception as e:
    print("❌ agent_step:", e)

try:
    _fl = [{"tool":"calculator"},{"tool":"none"},{"tool":"rag"}]
    assert abs(tool_choice_accuracy(_fl, ["calculator","rag","rag"]) - 2/3) < 1e-6
    print("✅ tool_choice_accuracy")
except Exception as e:
    print("❌ tool_choice_accuracy:", e)

try:
    _fq = [
        {"tool":"calculator","tool_output":{"result":4},"answer":"4"},
        {"tool":"calculator","tool_output":{"error":"bad"},"answer":"err"},
        {"tool":"rag","tool_output":{"passages":["p1"]},"answer":"..."},
        {"tool":"rag","tool_output":{"passages":[]},"answer":"..."},
        {"tool":"none","tool_output":None,"answer":"hello"},
        {"tool":"none","tool_output":None,"answer":""},
    ]
    assert abs(quality_proxy_score(_fq) - 3/6) < 1e-6
    print("✅ quality_proxy_score")
except Exception as e:
    print("❌ quality_proxy_score:", e)

print("Done.")